In [147]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy
import time

## Hyperparameter

In [131]:
sklearn_random_state = 42
torch_random_state = 42
train_ratio = 0.8
train_generator = torch.Generator().manual_seed(torch_random_state)
# this give tmp ratio as 0.2

# validation_ratio + test_ratio msut sum up to 1 but 

validation_ratio = 0.5 #50% of tmp ratio no train ratio
test_ratio = 0.5 #50% of tmp ratio no train ratio

model_huggingface_repo_name = "prajjwal1/bert-tiny"



# fine tuning config

max_training_step = 2

batch_size = 16
device = "cuda" if torch.cuda.is_available() else "cpu"
learning_rate = 1e-5
weight_decay = 0.01

min_delta = 0.001
patience = 20



## Data preprocessing

In [132]:
raw_data_path = "s3://mlops-project-bucket-602343785232-ap-southeast-1-an/raw_data/IT Support Ticket Data.csv"
print(raw_data_path)
raw_df = pd.read_csv(raw_data_path)

s3://mlops-project-bucket-602343785232-ap-southeast-1-an/raw_data/IT Support Ticket Data.csv


In [133]:
print(f"before: {len(raw_df):,}")
raw_df.dropna(subset=["Body"],inplace=True)
raw_df.drop_duplicates(subset="Body",inplace=True)
raw_df.reset_index(drop=True,inplace=True)
print(f"after: {len(raw_df):,}")

before: 29,651
after: 25,055


## load model

In [134]:

tokenizer = BertTokenizer.from_pretrained(model_huggingface_repo_name)
bert = BertModel.from_pretrained(model_huggingface_repo_name)

Loading weights: 100%|██████████| 39/39 [00:00<00:00, 12392.26it/s]
[transformers] BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [135]:
bert

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-1): 2 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=128, out_features=128, bias=True)
            (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.

In [136]:
bert.pooler.dense

Linear(in_features=128, out_features=128, bias=True)

## data partitioning and dataset build

In [137]:
class TicketPriorityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer,label_onehot_id_map):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.label_onehot_id_map = label_onehot_id_map

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt",
        )

        item = {key: value.squeeze(0) for key, value in encoded.items()}

        one_hot_index = label_onehot_id_map[self.labels[idx]]
        one_hot = np.eye(3)[one_hot_index]
        item["labels"] = torch.tensor(one_hot, dtype=torch.long)
        return item

#### partioning data

In [138]:
selected_columns = ['Body','stratify_group','Priority','Department']
raw_df["stratify_group"] = raw_df["Department"] + "_" + raw_df["Priority"]

train_df, tmp_df = train_test_split(
    raw_df[selected_columns],
    train_size=train_ratio,
    random_state=42,
    stratify=raw_df["stratify_group"]
)

valid_df, test_df = train_test_split(
    tmp_df,
    test_size=validation_ratio,
    random_state=42,
    stratify=tmp_df["stratify_group"]
)

In [139]:
# build datasets
label_onehot_id_map = {"low":0,"medium":1,"high":2}
train_dataset = TicketPriorityDataset(
    train_df["Body"],
    train_df["Priority"],
    tokenizer,
    label_onehot_id_map
)
valid_dataset = TicketPriorityDataset(
    valid_df["Body"],
    valid_df["Priority"],
    tokenizer,
    label_onehot_id_map
)
test_dataset = TicketPriorityDataset(
    test_df["Body"],
    test_df["Priority"],
    tokenizer,
    label_onehot_id_map
)

## configurate dataloader

In [140]:


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=train_generator,
)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# view shape
batch = next(iter(train_loader))
for key, value in batch.items():
    print(key, value.shape)


input_ids torch.Size([16, 128])
token_type_ids torch.Size([16, 128])
attention_mask torch.Size([16, 128])
labels torch.Size([16, 3])


## Model modification

We modify the loaded BERT tiny pooler in place.

Original pooler:

```text
CLS token [batch_size, 128]
→ Linear(128 → 128)
→ Tanh()
→ pooler_output [batch_size, 128]
```

Modified pooler:

```text
CLS token [batch_size, 128]
→ Linear(128 → 3)
→ Identity()
→ logits [batch_size, 3]
```

`Identity()` removes the `Tanh()` effect while keeping the model edit simple and visible when printing `bert.pooler`.


In [ ]:

num_labels = 3

print("Before")
print(bert.pooler)

bert.pooler.dense = torch.nn.Linear(
    bert.config.hidden_size,
    num_labels,
)
bert.pooler.activation = torch.nn.Identity()

model = bert.to(device)

print("After")
print(bert.pooler)


Before
BertPooler(
  (dense): Linear(in_features=128, out_features=128, bias=True)
  (activation): Tanh()
)
After
BertPooler(
  (dense): Linear(in_features=128, out_features=3, bias=True)
  (activation): Identity()
)


In [142]:
bert

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-1): 2 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=128, out_features=128, bias=True)
            (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.

## early stopping module

In [143]:




class EarlyStopping:
    def __init__(self, min_delta, patience):
        self.min_delta = min_delta
        self.patience = patience
        self.best_step = None
        self.best_value = None
        self.best_model_state = None

    def check_early_stopping_status(self, value, step):
        if self.best_value is None:
            return "new_minimum"

        is_new_minimum = value < self.best_value - self.min_delta
        if is_new_minimum:
            return "new_minimum"

        steps_without_improvement = step - self.best_step
        if steps_without_improvement >= self.patience:
            return "stop"

        return "outside_threshold"

    def record_best_state(self, value, model, step):
        self.best_value = value
        self.best_step = step
        self.best_model_state = copy.deepcopy(model.state_dict())

    def return_best_model(self):
        return self.best_value, self.best_step, self.best_model_state



## pre train configuration

### 1. loss function
### 2. optimizer

In [144]:

loss_function = torch.nn.CrossEntropyLoss(reduction="sum")
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

print(f"device: {device}")
print(f"learning_rate: {learning_rate}")
print(f"weight_decay: {weight_decay}")


device: cpu
learning_rate: 1e-05
weight_decay: 0.01


## model finetuning

In [ ]:
### CODEX TASK -- {model finetuning}
import time

train_loss = []
valid_loss = []
train_time = []
valid_time = []
epoch_time = []

early_stopping = EarlyStopping(min_delta=min_delta, patience=patience)
training_start_time = time.perf_counter()

for step in range(max_training_step):
    epoch_start_time = time.perf_counter()

    running_train_loss = 0
    total_train_sample = 0
    train_start_time = time.perf_counter()

    model.train()
    for batch in train_loader:
        labels = batch.pop("labels").to(device)
        if labels.ndim == 2:
            labels = labels.argmax(dim=1)
        labels = labels.long()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        logits = outputs.pooler_output
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        batch_sample = labels.size(0)
        running_train_loss += loss.item()
        total_train_sample += batch_sample

    avg_train_loss = running_train_loss / total_train_sample
    train_duration = time.perf_counter() - train_start_time
    train_loss.append(avg_train_loss)
    train_time.append(train_duration)

    running_valid_loss = 0
    total_valid_sample = 0
    valid_start_time = time.perf_counter()

    model.eval()
    with torch.no_grad():
        for batch in valid_loader:
            labels = batch.pop("labels").to(device)
            if labels.ndim == 2:
                labels = labels.argmax(dim=1)
            labels = labels.long()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            logits = outputs.pooler_output
            loss = loss_function(logits, labels)

            batch_sample = labels.size(0)
            running_valid_loss += loss.item()
            total_valid_sample += batch_sample

    avg_valid_loss = running_valid_loss / total_valid_sample
    valid_duration = time.perf_counter() - valid_start_time
    epoch_duration = time.perf_counter() - epoch_start_time

    valid_loss.append(avg_valid_loss)
    valid_time.append(valid_duration)
    epoch_time.append(epoch_duration)

    early_stopping_status = early_stopping.check_early_stopping_status(avg_valid_loss, step)

    if early_stopping_status == "new_minimum":
        early_stopping.record_best_state(avg_valid_loss, model, step)

    print(
        f"step={step + 1}/{max_training_step} "
        f"train_loss={avg_train_loss:.4f} "
        f"valid_loss={avg_valid_loss:.4f} "
        f"train_time={train_duration:.1f}s "
        f"valid_time={valid_duration:.1f}s "
        f"epoch_time={epoch_duration:.1f}s "
        f"early_stopping_status={early_stopping_status}"
    )

    if early_stopping_status == "stop":
        model.load_state_dict(early_stopping.best_model_state)
        print(
            f"early stopped. "
            f"best_step={early_stopping.best_step + 1} "
            f"best_valid_loss={early_stopping.best_value:.4f}"
        )
        break

total_training_time = time.perf_counter() - training_start_time
print(f"total_training_time={total_training_time:.1f}s")
### END CODEX TASK